[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/magnetism.ipynb)

# Moving charges, magnetism and matter full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the course's worked problems) and **tier 3** stretch problems (run a formula
backwards, combine two ideas, spot the trap). Each tier is harder than the last.

This notebook also has the course's **simulations and sums**: a charge circling in a magnetic field, a wire's and a
loop's field added up piece by piece, Ampère's sum round a loop, a solenoid built from single loops, a multimeter's
ranges from one needle, and a magnet turning in a field. You can move all of them yourself: the charge's **speed** and
the **field**, the **number of pieces**, the **loop's place**, the **coil's length**, the meter's **two ranges**, and
the magnet's **angle**.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Give the answer
in the unit the question asks for, to two decimals or more. Big and small numbers in E notation: `1.6e-19` for
1.6 × 10⁻¹⁹, `4e-7*pi` for 4π × 10⁻⁷ (`1.6*10^-19` works too). You can type the arithmetic: `2e-7*12/0.03*1e6`,
`9.11e-31*3e7/(1.6e-19*5e-3)*100`, `sqrt(1e-3*0.02/2e-7)`. Numbers without thousands separators: `4998000`, not
`4,998,000`. For several answers, separate them with commas, in the order the question lists them (the box's label
shows the order). You can also put your answer between the quotes in a cell's last line, `practice("P1.4", "")`, and run
the cell: it checks straight away.

**The cheat sheet.** Every formula and constant from the course on two pages:
[moving charges, magnetism and matter cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/magnetism_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the constants, the simulator and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # charges and lines in chalk, labels dim, ghost lines
SKY, MINT, CORAL, VIOLET, GOLD = "#6fb6bb", "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# velocity SKY · gravity MINT · force CORAL · energy VIOLET · wires and field lines CHALK · the one mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the constants (the course's rounded values; the same in every question) ----
MU0 = 4 * math.pi * 1e-7   # T m/A, mu nought: four pi times ten to the minus seven
KM = 1e-7                  # T m/A, mu nought over four pi
E = 1.60e-19               # C, the size of one electron's (or proton's) charge
ME = 9.11e-31              # kg, an electron's mass
MP = 1.67e-27              # kg, a proton's mass
g = 9.8                    # m/s²

# ---- the course's simulator: fourth-order Runge–Kutta, a fixed step at a time ----
def simulate(deriv, state, t_end, dt=1e-3, stop=None):
    """deriv(t, s) -> ds/dt as a tuple; state a tuple of numbers. Returns the list of (t, state) samples from 0 to
    t_end, or up to the first step where stop(t, s) is True."""
    t, s = 0.0, tuple(float(v) for v in state)
    out = [(t, s)]
    for _ in range(int(round(t_end / dt))):
        k1 = deriv(t, s)
        k2 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k1)))
        k3 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k2)))
        k4 = deriv(t + dt, tuple(a + dt * b for a, b in zip(s, k3)))
        s = tuple(a + dt / 6 * (b1 + 2 * b2 + 2 * b3 + b4) for a, b1, b2, b3, b4 in zip(s, k1, k2, k3, k4))
        t = t + dt
        out.append((t, s))
        if stop is not None and stop(t, s): return out
    return out

def close(a, b, rel=1e-4):
    """True when a simulated or summed value matches the formula (to a ten-thousandth by default)."""
    return abs(a - b) <= rel * max(abs(a), abs(b))

# ---- the course's formulas ----
def sind(d): return math.sin(math.radians(d))
def cosd(d): return math.cos(math.radians(d))
def force_charge(q, v, B, theta=90.0): return q * v * B * sind(theta)                 # F = q v B sin θ
def radius(m, v, q, B): return m * v / (q * B)                                        # r = m v / (q B)
def period(m, q, B): return 2 * math.pi * m / (q * B)                                 # T = 2π m / (q B)
def force_wire(B, I, L, theta=90.0): return B * I * L * sind(theta)                   # F = B I L sin θ
def wire_field(I, r): return MU0 * I / (2 * math.pi * r)                              # B = μ₀ I / (2π r)
def loop_axis(I, R, x=0.0, N=1): return MU0 * N * I * R * R / (2 * (R * R + x * x) ** 1.5)   # B = μ₀ I R² / (2 s³)
def solenoid(n, I): return MU0 * n * I                                                # B = μ₀ n I
def parallel_force(I1, I2, d): return MU0 * I1 * I2 / (2 * math.pi * d)               # F/L = μ₀ I₁ I₂ / (2π d)
def moment(N, I, A): return N * I * A                                                 # m = N I A
def torque(m, B, theta=90.0): return m * B * sind(theta)                              # τ = m B sin θ
def dipole_axis(m, r): return KM * 2 * m / r ** 3                                     # B = (μ₀/4π) 2m / r³
def dipole_equator(m, r): return KM * m / r ** 3                                      # B = (μ₀/4π) m / r³
def dipole_energy(m, B, theta): return -m * B * cosd(theta)                           # U = −m B cos θ
def shunt(Ig, G, I): return Ig * G / (I - Ig)                                         # S = I_g G / (I − I_g)
def series_resistor(Ig, G, V): return V / Ig - G                                      # R = V / I_g − G

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "cos": math.cos, "sin": math.sin, "acos": math.acos}

def calc(text):
    """Plain arithmetic only: numbers (8.99e9 too), + − × ÷, powers, brackets, sqrt, cos, sin, acos (radians) and pi."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id in ("pi", "π"): return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = (r"\s*(N m|A m²|A m2|A/m|N/m|m/s|µT|uT|mT|T|µA|uA|mA|A|mN|N|Ω|ohms?|mV|V|ns|µs|s|cm|mm|m|mJ|J|%|°|"
        r"degrees?|times)\s*$")

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("√", "sqrt")
    t = re.sub(UNIT, "", s).strip()
    try: return calc(t)
    except Exception: return s

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Magnetic force',
     "q": 'A proton (charge 1.60 × 10⁻¹⁹ C) moves at 5.0 × 10⁵ m/s at right angles to a magnetic field of 0.20 T. What is the force on it, in units of 10⁻¹⁴ N?',
     "parts": ['force (× 10⁻¹⁴ N)'],
     "answer": [1.6], "tol": [0.005], "function": '—',
     "hint": 'at right angles sin θ = 1: F = q v B',
     "working": 'F = qvB = 1.60 × 10⁻¹⁹ × 5.0 × 10⁵ × 0.20 = 1.6 × 10⁻¹⁴ N'},
    {"id": 'P1.2', "tier": 1, "group": 'Magnetic force',
     "q": 'A straight wire 0.25 m long carries 8 A at right angles to a field of 0.4 T. What is the force on it, in newtons?',
     "parts": ['force (N)'],
     "answer": [0.8], "tol": [0.005], "function": '—',
     "hint": 'F = B I L',
     "working": 'F = BIL = 0.4 × 8 × 0.25 = 0.8 N'},
    {"id": 'P1.3', "tier": 1, "group": 'Magnetic force',
     "q": 'An electron (mass 9.11 × 10⁻³¹ kg, charge 1.60 × 10⁻¹⁹ C) moves at 3.0 × 10⁷ m/s at right angles to a field of 5.0 mT. What is the radius of its circle, in cm?',
     "parts": ['radius (cm)'],
     "answer": [3.41625], "tol": [0.005], "function": '—',
     "hint": 'the magnetic force is the inward force: qvB = mv²/r',
     "working": 'r = mv/(qB) = 9.11 × 10⁻³¹ × 3.0 × 10⁷ / (1.60 × 10⁻¹⁹ × 0.005) = 0.03416 m = 3.4162 cm'},
    {"id": 'P1.4', "tier": 1, "group": 'Fields from currents',
     "q": 'A long straight wire carries 12 A. What is the magnetic field 3 cm from it, in microtesla?',
     "parts": ['field (µT)'],
     "answer": [80.0], "tol": [0.005], "function": '—',
     "hint": 'μ₀/2π = 2 × 10⁻⁷, so B = 2e-7 × I ÷ r',
     "working": 'B = μ₀I/(2πr) = 2 × 10⁻⁷ × 12 / 0.03 = 8 × 10⁻⁵ T = 80 µT'},
    {"id": 'P1.5', "tier": 1, "group": 'Fields from currents',
     "q": 'A single circular loop of radius 8 cm carries 5 A. What is the field at its centre, in microtesla?',
     "parts": ['field (µT)'],
     "answer": [39.269908169872416], "tol": [0.005], "function": 'pi',
     "hint": 'at the centre, B = μ₀ I ÷ (2R)',
     "working": 'B = μ₀I/(2R) = 4π × 10⁻⁷ × 5 / 0.16 = 3.9270e-05 T = 39.2699 µT'},
    {"id": 'P1.6', "tier": 1, "group": 'Fields from currents',
     "q": 'A long solenoid has 1500 turns on every metre and carries 0.8 A. What is the field inside it, in millitesla?',
     "parts": ['field (mT)'],
     "answer": [1.5079644737231008], "tol": [0.005], "function": 'pi',
     "hint": 'B = μ₀ n I, with n in turns per metre',
     "working": 'B = μ₀nI = 4π × 10⁻⁷ × 1500 × 0.8 = 1.5080e-03 T = 1.5080 mT'},
    {"id": 'P1.7', "tier": 1, "group": 'Forces between currents',
     "q": 'Two long parallel wires 4 cm apart each carry 6 A. What is the force on each metre of either wire, in units of 10⁻⁴ N?',
     "parts": ['force per metre (× 10⁻⁴ N)'],
     "answer": [1.7999999999999998], "tol": [0.005], "function": '—',
     "hint": 'F/L = 2e-7 × I1 × I2 ÷ d',
     "working": 'F/L = μ₀I₁I₂/(2πd) = 2 × 10⁻⁷ × 36 / 0.04 = 1.8 × 10⁻⁴ N on each metre'},
    {"id": 'P1.8', "tier": 1, "group": 'Forces between currents',
     "q": 'A flat coil of 40 turns, each of area 0.003 m², carries 2 A. What is its magnetic moment, in A m²?',
     "parts": ['magnetic moment (A m²)'],
     "answer": [0.24], "tol": [0.005], "function": '—',
     "hint": 'magnetic moment = turns × current × area',
     "working": 'm = NIA = 40 × 2 × 0.003 = 0.24 A m²'},
    {"id": 'P1.9', "tier": 1, "group": 'The galvanometer',
     "q": "A galvanometer's coil has a resistance of 100 Ω and reads full scale at 1 mA. What voltage is across the coil at full scale, in volts?",
     "parts": ['voltage (V)'],
     "answer": [0.1], "tol": [0.005], "function": '—',
     "hint": "a resistor's voltage is its current times its resistance",
     "working": 'V = I × R = 0.001 × 100 = 0.1 V'},
    {"id": 'P1.10', "tier": 1, "group": 'Magnetism in matter',
     "q": 'A long coil with 800 turns on every metre carries 1.5 A. What is the magnetic intensity H inside it, in A/m?',
     "parts": ['H (A/m)'],
     "answer": [1200.0], "tol": [0.005], "function": '—',
     "hint": "H is set by the coil's current alone: n × I",
     "working": 'H = nI = 800 × 1.5 = 1200 A/m (whatever fills the coil)'},
    {"id": 'P2.1', "tier": 2, "group": 'Magnetic force',
     "q": 'A proton (mass 1.67 × 10⁻²⁷ kg, charge 1.60 × 10⁻¹⁹ C) circles in a field of 0.50 T. How long does one lap take, in nanoseconds? (Does it depend on the speed?)',
     "parts": ['lap time (ns)'],
     "answer": [131.16149328737387], "tol": [0.005], "function": 'pi',
     "hint": 'T = 2π m ÷ (q B)',
     "working": 'T = 2πm/(qB) = 2π × 1.67 × 10⁻²⁷ / (1.60 × 10⁻¹⁹ × 0.50) = 1.3116e-07 s = 131.1615 ns; the speed cancels'},
    {"id": 'P2.2', "tier": 2, "group": 'Magnetic force',
     "q": 'An electric field of 4500 V/m and a magnetic field of 0.03 T are set at right angles, and a beam is fired at right angles to both. What speed passes straight through, in units of 10⁵ m/s?',
     "parts": ['speed (× 10⁵ m/s)'],
     "answer": [1.5], "tol": [0.005], "function": '/',
     "hint": 'the two forces cancel when v = E ÷ B',
     "working": 'qE = qvB, so v = E/B = 4500 / 0.03 = 1.5 × 10⁵ m/s'},
    {"id": 'P2.3', "tier": 2, "group": 'Fields from currents',
     "q": 'A single loop of radius 3 cm carries 2 A. What is the field on its axis, 4 cm from its centre, in microtesla?',
     "parts": ['field (µT)'],
     "answer": [9.047786842338603], "tol": [0.005], "function": 'pi',
     "hint": 'B = μ₀ I R² ÷ (2 s³), with s the distance from the wire to the point',
     "working": 'each piece is √(3² + 4²) = 5 cm away: B = μ₀IR²/(2s³) = 4π × 10⁻⁷ × 2 × 0.0009 / (2 × 0.000125) = 9.0478 µT'},
    {"id": 'P2.4', "tier": 2, "group": 'Fields from currents',
     "q": 'A wire of radius 2 mm carries 10 A, spread evenly. What is the field 1 mm from its centre, inside the wire (mT), and 4 mm from its centre, outside (mT)?',
     "parts": ['inside, at 1 mm (mT)', 'outside, at 4 mm (mT)'],
     "answer": [0.5, 0.5], "tol": [0.005, 0.005], "function": '—',
     "hint": 'only the current inside your loop counts',
     "working": 'inside, a loop of half the radius holds a quarter of the current: B = 2 × 10⁻⁷ × 2.5 / 0.001 = 0.5 mT; outside: 2 × 10⁻⁷ × 10 / 0.004 = 0.5 mT'},
    {"id": 'P2.5', "tier": 2, "group": 'Forces between currents',
     "q": 'Two long parallel wires 5 cm apart carry 15 A and 20 A in the same direction. What is the force on each metre, in millinewtons? (Do they attract or repel?)',
     "parts": ['force per metre (mN)'],
     "answer": [1.2000000000000002], "tol": [0.005], "function": '—',
     "hint": 'F/L = μ₀ I₁ I₂ ÷ (2π d)',
     "working": 'F/L = 2 × 10⁻⁷ × 15 × 20 / 0.05 = 1.2 × 10⁻³ N = 1.2 mN on each metre; currents the same way attract'},
    {"id": 'P2.6', "tier": 2, "group": 'Forces between currents',
     "q": 'A coil of 50 turns, 4 cm by 5 cm, carries 1 A. Its magnetic moment makes 30° with a field of 0.3 T. What is the torque on it, in N m?',
     "parts": ['torque (N m)'],
     "answer": [0.014999999999999998], "tol": [0.005], "function": 'sin',
     "hint": 'torque = m B sin θ, with m = N I A',
     "working": 'm = NIA = 50 × 1 × 0.002 = 0.1 A m²; τ = mB sin θ = 0.1 × 0.3 × 0.5 = 0.015 N m'},
    {"id": 'P2.7', "tier": 2, "group": 'The galvanometer',
     "q": 'A galvanometer has a coil of 100 Ω and reads full scale at 1 mA. What shunt turns it into an ammeter reading 5 A full scale, in ohms? (Give four decimals.)',
     "parts": ['shunt (Ω)'],
     "answer": [0.020004000800160033], "tol": [5e-05], "function": '—',
     "hint": 'the shunt and the coil have the same voltage across them',
     "working": 'S = I_g G/(I − I_g) = 0.001 × 100 / (5 − 0.001) = 0.020004 Ω'},
    {"id": 'P2.8', "tier": 2, "group": 'The galvanometer',
     "q": 'The same galvanometer (100 Ω, 1 mA full scale) is to read 15 V full scale. What series resistor does it need, in ohms?',
     "parts": ['series resistor (Ω)'],
     "answer": [14900.0], "tol": [0.005], "function": '—',
     "hint": 'at full scale, the whole 15 V drives 1 mA through R and the coil',
     "working": 'R = V/I_g − G = 15 / 0.001 − 100 = 14,900 Ω'},
    {"id": 'P2.9', "tier": 2, "group": 'Magnets',
     "q": 'A bar magnet has a magnetic moment of 0.8 A m². What is its field 20 cm from its centre, on its axis (µT), and at the same distance on the line through its middle, square to the axis (µT)?',
     "parts": ['on the axis (µT)', 'on the side line (µT)'],
     "answer": [19.999999999999996, 9.999999999999998], "tol": [0.005, 0.005], "function": '—',
     "hint": 'axis: (μ₀/4π) 2m ÷ r³; side: (μ₀/4π) m ÷ r³',
     "working": 'axis: 10⁻⁷ × 2 × 0.8 / 0.2³ = 2 × 10⁻⁵ T = 20 µT; side line: half of that, 10 µT, pointing the opposite way'},
    {"id": 'P2.10', "tier": 2, "group": 'Magnetism in matter',
     "q": 'A coil with 1200 turns on every metre carries 0.5 A and is filled with iron of relative permeability 600. What is the field B inside, in tesla?',
     "parts": ['B (T)'],
     "answer": [0.4523893421169302], "tol": [0.005], "function": 'pi',
     "hint": 'B = μ₀ μᵣ H, with H = n I',
     "working": 'H = nI = 600 A/m; B = μ₀μᵣH = 4π × 10⁻⁷ × 600 × 600 = 0.4524 T'},
    {"id": 'P3.1', "tier": 3, "group": 'Magnetic force',
     "q": 'A straight wire of mass 2 g and length 10 cm lies level, at right angles to a level magnetic field, and carries 10 A. What field holds it up against gravity (g = 9.8 m/s²), in millitesla?',
     "parts": ['field (mT)'],
     "answer": [19.6], "tol": [0.005], "function": '—',
     "hint": 'set the magnetic force equal to the weight',
     "working": 'BIL = mg, so B = mg/(IL) = 0.002 × 9.8 / (10 × 0.10) = 0.0196 T = 19.6 mT'},
    {"id": 'P3.2', "tier": 3, "group": 'Magnetic force',
     "q": 'An electron (mass 9.11 × 10⁻³¹ kg, charge 1.60 × 10⁻¹⁹ C) enters a field of 2.0 mT at 1.0 × 10⁷ m/s, at 30° to the field. What is the radius of its helix (cm), and how far does it move along the field in one lap (cm)?',
     "parts": ['radius (cm)', 'advance per lap (cm)'],
     "answer": [1.4234375, 15.490980195162871], "tol": [0.005, 0.005], "function": 'sin, cos',
     "hint": 'split the velocity: across the field (the circle) and along it (the drift)',
     "working": 'across the field: v sin 30° = 5.0 × 10⁶ m/s, r = 1.4234 cm; along it: v cos 30° × T = 8.66 × 10⁶ × 1.7887e-08 = 15.4910 cm'},
    {"id": 'P3.3', "tier": 3, "group": 'Fields from currents',
     "q": "At what distance from a long wire carrying 5 A does its field equal 50 µT (about the Earth's), in cm?",
     "parts": ['distance (cm)'],
     "answer": [2.0], "tol": [0.005], "function": '—',
     "hint": 'run B = μ₀I/(2πr) backwards',
     "working": 'r = μ₀I/(2πB) = 2 × 10⁻⁷ × 5 / 50 × 10⁻⁶ = 0.02 m = 2 cm'},
    {"id": 'P3.4', "tier": 3, "group": 'Fields from currents',
     "q": 'A long solenoid has 2500 turns on every metre. What current gives 10 mT inside it, in amps?',
     "parts": ['current (A)'],
     "answer": [3.1830988618379066], "tol": [0.005], "function": 'pi',
     "hint": 'run B = μ₀ n I backwards',
     "working": 'I = B/(μ₀n) = 0.010 / (4π × 10⁻⁷ × 2500) = 3.1831 A'},
    {"id": 'P3.5', "tier": 3, "group": 'Forces between currents',
     "q": 'Two long parallel wires 2 cm apart carry equal currents. Each metre feels a force of 1.0 × 10⁻³ N. What is the current in each, in amps?',
     "parts": ['current (A)'],
     "answer": [10.0], "tol": [0.005], "function": 'sqrt',
     "hint": 'F/L = 2e-7 × I² ÷ d; solve for I',
     "working": 'I² = (F/L) × 2πd/μ₀ = 10⁻³ × 0.02 / 2 × 10⁻⁷ = 100, so I = 10 A'},
    {"id": 'P3.6', "tier": 3, "group": 'The galvanometer',
     "q": 'A meter movement reads full scale at 50 µA and its coil has 2000 Ω. What shunt makes a 10 A range (Ω, four decimals), and what series resistor makes a 250 V range (Ω)?',
     "parts": ['shunt (Ω)', 'series resistor (Ω)'],
     "answer": [0.010000050000250001, 4998000.0], "tol": [5e-05, 0.005], "function": '—',
     "hint": '0.1 V across the coil at full scale, either way',
     "working": 'S = 50 × 10⁻⁶ × 2000 / (10 − 0.00005) = 0.0100001 Ω; R = 250 / 50 × 10⁻⁶ − 2000 = 4,998,000 Ω'},
    {"id": 'P3.7', "tier": 3, "group": 'Magnets',
     "q": 'Turning a small magnet from lined up with a 0.3 T field to lined up against it takes 0.012 J. What is its magnetic moment, in A m²?',
     "parts": ['magnetic moment (A m²)'],
     "answer": [0.02], "tol": [0.005], "function": '—',
     "hint": 'U = −m B cos θ: from −mB up to +mB',
     "working": 'U goes from −mB to +mB: W = 2mB, so m = 0.012 / (2 × 0.3) = 0.02 A m²'},
    {"id": 'P3.8', "tier": 3, "group": 'Magnetism in matter',
     "q": "An iron ring is wound with a coil. With H = 1500 A/m the field in the iron is 0.9 T. What is the iron's relative permeability, and what is its magnetisation M, in units of 10⁵ A/m?",
     "parts": ['relative permeability', 'M (× 10⁵ A/m)'],
     "answer": [477.46482927568604, 7.146972439135291], "tol": [0.005, 0.005], "function": 'pi',
     "hint": 'B = μ₀ μᵣ H, and M = (μᵣ − 1) H',
     "working": 'μᵣ = B/(μ₀H) = 0.9 / (4π × 10⁻⁷ × 1500) = 477.4648; M = (μᵣ − 1)H = 714697.2 A/m'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else "✓ all right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
            print(textwrap.fill(f"Hint: {q['hint']}", 110))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

assert abs(MU0 / (4 * math.pi) - KM) < 1e-20
print("Ready: mu nought =", MU0, "T m/A; e =", E, "C")

## Tier 1 · Warm-up

One idea, one step: the force on a moving charge and on a wire, the radius of a charge's circle, the field of a long
wire, a loop and a solenoid, the force between two wires, a coil's magnetic moment, the voltage across a meter's coil,
and the magnetic intensity H in a coil.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one short function in the first cell: `force_charge(q, v, B)`, `radius(m, v, q, B)`,
`wire_field(I, r)`, `loop_axis(I, R, x)`, `solenoid(n, I)`, `parallel_force(I1, I2, d)`, `moment(N, I, A)` and the
rest. Work in tesla, amps, metres and kilograms (µT is `1e-6`, mT is `1e-3`, cm is `0.01`), then change the unit at the
end. Each line below computes one warm-up answer and checks it against the practice set.

In [ ]:
print(f"P1.3  r = {radius(ME, 3e7, E, 5e-3) * 100:.2f} cm;  P1.5  B = {loop_axis(5, 0.08) * 1e6:.2f} µT")
assert agrees("P1.1", [force_charge(E, 5e5, 0.2) / 1e-14]) and agrees("P1.2", [force_wire(0.4, 8, 0.25)])
assert agrees("P1.3", [radius(ME, 3e7, E, 5e-3) * 100]) and agrees("P1.4", [wire_field(12, 0.03) * 1e6])
assert agrees("P1.5", [loop_axis(5, 0.08) * 1e6]) and agrees("P1.6", [solenoid(1500, 0.8) * 1e3])
assert agrees("P1.7", [parallel_force(6, 6, 0.04) / 1e-4]) and agrees("P1.8", [moment(40, 2, 0.003)])
assert agrees("P1.9", [0.001 * 100]) and agrees("P1.10", [800 * 1.5])
print("Every tier 1 answer, worked out in Python, matches the practice set.")

## A charge circling in a magnetic field

An electron is fired straight across a steady field and moved step by step, with nothing but the force q v B, always at
right angles to its motion. The path closes into a circle. Change the speed and the field: the radius follows
m v / (q B), but the lap time, 2π m / (q B), does not care about the speed at all. The speed itself never changes: a
magnetic force does no work.

In [ ]:
def circle_run(v, B, m=ME, q=E, laps=1.0, steps=1500):
    w = q * B / m                                          # the turning rate the force gives
    T = 2 * math.pi / w
    return simulate(lambda t, s: (s[2], s[3], w * s[3], -w * s[2]), (0.0, 0.0, v, 0.0), laps * T, T / steps), T

def circle_picture(v_million=10.0, B_mT=1.0):
    v, B = v_million * 1e6, B_mT * 1e-3
    run, T = circle_run(v, B)
    xs = np.array([s[0] for _, s in run]) * 100; ys = np.array([s[1] for _, s in run]) * 100
    r_sim = (ys.max() - ys.min()) / 2
    speed_change = max(abs(math.hypot(s[2], s[3]) / v - 1) for _, s in run)
    fig, ax = plt.subplots(figsize=(6.4, 6.0))
    fig.subplots_adjust(left=0.14, right=0.97, top=0.86, bottom=0.11)
    ax.plot(xs, ys, color=GOLD); ax.plot([0], [0], "o", color=CHALK)
    lim = max(13.0, 1.15 * 2 * r_sim); ax.set_xlim(-lim / 2, lim / 2); ax.set_ylim(-lim, 0.08 * lim)
    ax.set_aspect("equal"); ax.set_xlabel("cm"); ax.set_ylabel("cm")
    fig.suptitle(f"r = {r_sim:.2f} cm (m v / q B = {radius(ME, v, E, B) * 100:.2f} cm) · one lap: {T * 1e9:.1f} ns",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig, f"simulated · speed changed by {speed_change:.0e}"); plt.show()
    back = math.hypot(run[-1][1][0], run[-1][1][1])       # after the formula's lap time: how far from where it began?
    return r_sim / 100, T, speed_change, back

r_s, T_s, dv_s, back_s = circle_picture(10.0, 1.0)
assert close(r_s, radius(ME, 1e7, E, 1e-3), 1e-4) and back_s < 1e-4 * r_s and dv_s < 1e-8      # one lap in 2π m / (q B)
r_f, T_f, _, back_f = circle_picture(20.0, 1.0)
assert close(r_f, 2 * r_s, 1e-4) and T_f == T_s and back_f < 1e-4 * r_f     # twice as fast: twice the circle, the same lap time

In [ ]:
interact(circle_picture, v_million=FloatSlider(value=10.0, min=2.0, max=30.0, step=1.0, description="v (10⁶ m/s)"),
         B_mT=FloatSlider(value=1.0, min=0.5, max=5.0, step=0.5, description="B (mT)"));

## Tier 2 · Core

The level of the course's worked problems: a proton's lap time, crossed fields, a loop's axis, a thick wire, two wires,
the torque on a coil, a shunt and a series resistor, a bar magnet's field, and a coil with an iron core.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

The lap time has no speed in it. On a loop's axis, use the distance from the wire to the point, s = √(R² + x²). Inside a
thick wire, only the current inside your loop counts. A shunt and its coil share one voltage; a series resistor and its
coil share one current.

In [ ]:
assert agrees("P2.1", [period(MP, E, 0.5) * 1e9]) and agrees("P2.2", [4500 / 0.03 / 1e5])
print(f"P2.3  B = {loop_axis(2, 0.03, 0.04) * 1e6:.2f} µT (every piece is {math.hypot(0.03, 0.04) * 100:.0f} cm away)")
assert agrees("P2.3", [loop_axis(2, 0.03, 0.04) * 1e6])
inside = wire_field(10 * (1e-3 / 2e-3) ** 2, 1e-3)               # a loop of half the radius holds a quarter of the current
assert agrees("P2.4", [inside * 1e3, wire_field(10, 4e-3) * 1e3]) and agrees("P2.5", [parallel_force(15, 20, 0.05) * 1e3])
assert agrees("P2.6", [torque(moment(50, 1, 0.04 * 0.05), 0.3, 30)])
assert agrees("P2.7", [shunt(1e-3, 100, 5)]) and agrees("P2.8", [series_resistor(1e-3, 100, 15)])
assert agrees("P2.9", [dipole_axis(0.8, 0.2) * 1e6, dipole_equator(0.8, 0.2) * 1e6])
assert agrees("P2.10", [MU0 * 600 * 1200 * 0.5])
print("Every tier 2 answer, worked out in Python, matches the practice set.")

## A wire's field and a loop's field, added up piece by piece

The Biot–Savart law gives the small field of one short piece of current. Cut a long straight wire into pieces, add
every piece's field at a point 5 cm away, and the sum lands on μ₀ I / (2π r). Slide the number of pieces and the wire's
length: a short wire, or too few pieces, falls short. The second picture does the same for a loop, along its centre
line, and draws the formula μ₀ I R² / (2 s³) under the sum.

In [ ]:
def wire_sum(I=10.0, r=0.05, half=100.0, n=200000):
    z = (np.arange(n) + 0.5) * (2 * half / n) - half          # each piece's middle
    return float(np.sum(KM * I * (2 * half / n) * r / (z * z + r * r) ** 1.5))

def wire_picture(pieces=2000, half_length_m=1.0):
    ns = np.unique(np.geomspace(10, max(20, pieces), 40).astype(int))
    sums = [wire_sum(n=int(k), half=half_length_m) * 1e6 for k in ns]
    fig, ax = plt.subplots(figsize=(10, 4.0))
    fig.subplots_adjust(left=0.1, right=0.98, top=0.82, bottom=0.18)
    ax.semilogx(ns, sums, color=GOLD, label="added up, piece by piece")
    ax.axhline(wire_field(10, 0.05) * 1e6, color=DIM, ls="--", label="μ₀ I / (2π r)")
    ax.set_xlabel("pieces"); ax.set_ylabel("B (µT)"); ax.legend(frameon=False, loc="lower right")
    fig.suptitle(f"10 A, 5 cm away, a wire {2 * half_length_m:g} m long in {pieces} pieces: {sums[-1]:.3f} µT "
                 f"(a long wire: {wire_field(10, 0.05) * 1e6:.3f} µT)", x=0.03, ha="left", color=GOLD)
    footnote(fig, "summed piece by piece (Biot–Savart)"); plt.show()
    return sums[-1] * 1e-6

assert close(wire_sum(), wire_field(10, 0.05), 1e-5)                 # 200,000 pieces of a 200 m wire: 40 µT
wire_picture(2000, 1.0)

In [ ]:
interact(wire_picture, pieces=IntSlider(value=2000, min=20, max=20000, step=20, description="pieces"),
         half_length_m=FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05, description="half-length (m)"));

In [ ]:
def loop_axis_sum(I, R, x, n=720):
    ph = (np.arange(n) + 0.5) * 2 * np.pi / n
    dl = np.stack([-np.sin(ph), np.cos(ph), np.zeros(n)], 1) * (2 * np.pi * R / n)      # each piece, as an arrow
    rv = np.stack([-R * np.cos(ph), -R * np.sin(ph), np.full(n, x)], 1)                  # from the piece to the point
    b = np.sum(KM * I * np.cross(dl, rv) / (np.sum(rv * rv, 1) ** 1.5)[:, None], 0)
    return float(b[2]), float(np.hypot(b[0], b[1]))                                      # along the line, and sideways

def loop_picture(I=2.0, R_cm=5.0):
    R = R_cm / 100
    xs = np.linspace(0, 0.3, 61)
    sums = np.array([loop_axis_sum(I, R, x)[0] for x in xs]) * 1e6
    fig, ax = plt.subplots(figsize=(10, 4.0))
    fig.subplots_adjust(left=0.1, right=0.98, top=0.82, bottom=0.18)
    ax.plot(xs * 100, [loop_axis(I, R, x) * 1e6 for x in xs], color=DIM, lw=5, label="μ₀ I R² / (2 s³)")
    ax.plot(xs * 100, sums, color=GOLD, lw=1.6, label="720 pieces, added as arrows")
    ax.set_xlabel("distance along the centre line (cm)"); ax.set_ylabel("B (µT)"); ax.legend(frameon=False)
    fig.suptitle(f"{I:g} A in a loop of radius {R_cm:g} cm: {sums[0]:.2f} µT at the centre", x=0.03, ha="left", color=GOLD)
    footnote(fig, "summed piece by piece (Biot–Savart)"); plt.show()
    return sums[0] * 1e-6

assert close(loop_picture(2.0, 5.0), loop_axis(2, 0.05), 1e-9)
assert loop_axis_sum(2, 0.05, 0.05)[1] < 1e-15                       # the sideways parts cancel

In [ ]:
interact(loop_picture, I=FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description="I (A)"),
         R_cm=FloatSlider(value=5.0, min=2.0, max=15.0, step=1.0, description="R (cm)"));

## Ampère's sum round a loop you can move

A long wire carries 10 A straight out of the screen at the origin. Walk round a square loop, adding the field along each
small step times the step's length. While the wire passes through the loop the total is μ₀ I, whatever the loop's size
and wherever it sits. Slide the loop sideways until it misses the wire: the total drops to zero.

In [ ]:
def circulation(cx, cy, side, I=10.0, n=2000):
    h = side / 2
    c = [(cx - h, cy - h), (cx + h, cy - h), (cx + h, cy + h), (cx - h, cy + h)]
    tot = 0.0
    for k in range(4):
        (x0, y0), (x1, y1) = c[k], c[(k + 1) % 4]
        for j in range(n):
            mx, my = x0 + (x1 - x0) * (j + 0.5) / n, y0 + (y1 - y0) * (j + 0.5) / n
            d = math.hypot(mx, my); b = wire_field(I, d)
            tot += b * (-my / d) * (x1 - x0) / n + b * (mx / d) * (y1 - y0) / n
    return tot

def ampere_picture(centre_x_cm=0.0, side_cm=20.0):
    cx, side = centre_x_cm / 100, side_cm / 100
    total = circulation(cx, 0.003, side) / (MU0 * 10)
    fig, ax = plt.subplots(figsize=(6.6, 5.6))
    fig.subplots_adjust(left=0.12, right=0.97, top=0.86, bottom=0.11)
    for rr in (4, 8, 12, 16, 20):                                   # the wire's field lines: circles
        th = np.linspace(0, 2 * np.pi, 200); ax.plot(rr * np.cos(th), rr * np.sin(th), color=GHOST, lw=1)
    ax.plot([0], [0], "o", color=CHALK, ms=9)
    h = side_cm / 2
    ax.plot([centre_x_cm - h, centre_x_cm + h, centre_x_cm + h, centre_x_cm - h, centre_x_cm - h],
            [0.3 - h, 0.3 - h, 0.3 + h, 0.3 + h, 0.3 - h], color=GOLD)
    ax.set_xlim(-35, 35); ax.set_ylim(-28, 28); ax.set_aspect("equal"); ax.set_xlabel("cm"); ax.set_ylabel("cm")
    ax.grid(False)
    fig.suptitle(f"the field added round the loop = {total:.4f} × μ₀ I", x=0.03, ha="left", color=GOLD)
    footnote(fig, "summed round the loop, 8,000 steps"); plt.show()
    return total

assert close(ampere_picture(0.0, 20.0), 1.0, 1e-4) and close(ampere_picture(6.0, 20.0), 1.0, 1e-4)
assert abs(ampere_picture(25.0, 20.0)) < 1e-4                        # the loop misses the wire: nothing

In [ ]:
interact(ampere_picture, centre_x_cm=FloatSlider(value=0.0, min=-25.0, max=25.0, step=1.0, description="loop's centre (cm)"),
         side_cm=FloatSlider(value=20.0, min=6.0, max=30.0, step=2.0, description="side (cm)"));

## A solenoid built from single loops

A solenoid is a row of loops. Add every loop's field along the centre line, and the middle of a long coil reaches
μ₀ n I; at the very end it falls to about a half. Make the coil shorter with the same turns per metre: the middle sags
below μ₀ n I, because a short coil is not "long".

In [ ]:
def solenoid_sum(z, turns=400, length=0.20, R=0.01, I=2.0):
    zs = -length / 2 + length * (np.arange(turns) + 0.5) / turns
    return float(np.sum(MU0 * I * R * R / (2 * (R * R + (z - zs) ** 2) ** 1.5)))

def solenoid_picture(length_cm=20.0, turns_per_m=2000.0):
    length = length_cm / 100; turns = max(2, int(round(turns_per_m * length)))
    zs = np.linspace(-0.2, 0.2, 201)
    B = np.array([solenoid_sum(z, turns, length) for z in zs]) / solenoid(turns / length, 2.0)
    fig, ax = plt.subplots(figsize=(10, 4.0))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.82, bottom=0.18)
    ax.plot(zs * 100, B, color=GOLD); ax.axhline(1.0, color=DIM, ls="--"); ax.axhline(0.5, color=GHOST, ls="--")
    ax.axvspan(-length_cm / 2, length_cm / 2, color=GHOST, alpha=0.35)
    ax.set_xlabel("position along the centre line (cm); the coil is shaded"); ax.set_ylabel("B ÷ μ₀ n I")
    mid, end = solenoid_sum(0.0, turns, length) / solenoid(turns / length, 2.0), solenoid_sum(length / 2, turns, length) / solenoid(turns / length, 2.0)
    fig.suptitle(f"{turns} loops over {length_cm:g} cm (radius 1 cm): middle {mid:.3f}, end {end:.3f} of μ₀ n I",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig, "every loop's field added"); plt.show()
    return mid, end

mid_s, end_s = solenoid_picture(20.0, 2000.0)
assert close(mid_s, 10 / math.sqrt(101), 1e-3) and close(end_s, 20 / (2 * math.sqrt(401)), 2e-3)

In [ ]:
interact(solenoid_picture, length_cm=FloatSlider(value=20.0, min=2.0, max=30.0, step=1.0, description="length (cm)"),
         turns_per_m=FloatSlider(value=2000.0, min=500.0, max=4000.0, step=250.0, description="turns per m"));

## Tier 3 · Stretch

Run a formula backwards, combine two ideas, spot the trap: a wire held up by a field, a helix, a wire's field matched
to the Earth's, a solenoid's current, two wires' current from their force, a multimeter's two ranges, a magnet's moment
from the work to turn it, and an iron ring.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

Running a formula backwards is one line of algebra, then one line of Python. For the helix, split the velocity: the part
across the field makes the circle, the part along it carries the charge forward.

In [ ]:
assert agrees("P3.1", [0.002 * g / (10 * 0.10) * 1e3])
v_across, v_along = 1e7 * math.sin(math.radians(30)), 1e7 * math.cos(math.radians(30))
print(f"P3.2  r = {radius(ME, v_across, E, 2e-3) * 100:.2f} cm, advance = {v_along * period(ME, E, 2e-3) * 100:.2f} cm a lap")
assert agrees("P3.2", [radius(ME, v_across, E, 2e-3) * 100, v_along * period(ME, E, 2e-3) * 100])
assert agrees("P3.3", [MU0 * 5 / (2 * math.pi * 50e-6) * 100]) and agrees("P3.4", [0.010 / (MU0 * 2500)])
assert agrees("P3.5", [math.sqrt(1e-3 * 2 * math.pi * 0.02 / MU0)])
assert agrees("P3.6", [shunt(50e-6, 2000, 10), series_resistor(50e-6, 2000, 250)]) and agrees("P3.7", [0.012 / (2 * 0.3)])
mu_r = 0.9 / (MU0 * 1500)
assert agrees("P3.8", [mu_r, (mu_r - 1) * 1500 / 1e5])
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## A multimeter's ranges from one needle

The course's meter: a needle that reads full scale at 50 µA, on a coil of 2000 Ω, so 0.1 V across the coil at full
scale. A **shunt** beside the coil carries the rest of the current for an amp range; a **series resistor** holds the
current to 50 µA for a volt range. Slide the two ranges and watch the two resistors change. Every volt of range needs
20,000 Ω in all: one over 50 µA.

In [ ]:
def meter_picture(amps=10.0, volts=250.0, Ig_uA=50.0, G=2000.0):
    Ig = Ig_uA * 1e-6
    S, R = shunt(Ig, G, amps), series_resistor(Ig, G, volts)
    coil_current = amps * S / (S + G)                                # how any two resistors side by side share a current
    volt_current = volts / (R + G)                                   # the same current through the coil and its resistor
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.78, bottom=0.2, wspace=0.3)
    rng_i = np.geomspace(1e-3, 20, 80); rng_v = np.geomspace(0.5, 1000, 80)
    axes[0].loglog(rng_i, [shunt(Ig, G, i) for i in rng_i], color=DIM); axes[0].plot([amps], [S], "o", color=GOLD, ms=9)
    axes[0].set_xlabel("amp range (A)"); axes[0].set_ylabel("shunt (Ω)")
    axes[1].loglog(rng_v, [series_resistor(Ig, G, v) for v in rng_v], color=DIM); axes[1].plot([volts], [R], "o", color=GOLD, ms=9)
    axes[1].set_xlabel("volt range (V)"); axes[1].set_ylabel("series resistor (Ω)")
    fig.suptitle(f"{amps:g} A: shunt {S:.5g} Ω (coil takes 1 part in {amps / Ig:,.0f}) · {volts:g} V: series {R:,.0f} Ω "
                 f"({(R + G) / volts:,.0f} Ω per volt)", x=0.03, ha="left", color=GOLD)
    footnote(fig, "our own example · computed"); plt.show()
    return S, R, coil_current, volt_current

S_m, R_m, i_coil, i_volt = meter_picture(10.0, 250.0)
assert close(i_coil, 50e-6, 1e-9) and close(i_volt, 50e-6, 1e-9)     # full scale exactly, both ways
assert agrees("P3.6", [S_m, R_m])

In [ ]:
interact(meter_picture, amps=FloatSlider(value=10.0, min=0.01, max=20.0, step=0.01, description="amp range"),
         volts=FloatSlider(value=250.0, min=1.0, max=1000.0, step=1.0, description="volt range"),
         Ig_uA=FloatSlider(value=50.0, min=10.0, max=1000.0, step=10.0, description="full scale (µA)"),
         G=FloatSlider(value=2000.0, min=50.0, max=5000.0, step=50.0, description="coil (Ω)"));

## A magnet turning in a field

A small magnet (or a current loop) of magnetic moment m sits at an angle θ to a steady field. The torque on it is
m B sin θ. Turning it against that torque, a small angle at a time, stores energy: add up the small turns from 90° and
you get −m B cos θ. The magnet settles at the bottom of that curve, lined up with the field.

In [ ]:
def energy_sum(theta_deg, m, B, n=5000):
    a0, a1 = math.pi / 2, math.radians(theta_deg); h = (a1 - a0) / n
    return sum(m * B * math.sin(a0 + (j + 0.5) * h) * h for j in range(n))     # torque × small angle, added

def magnet_picture(m=0.4, B_mT=20.0, theta=60.0):
    B = B_mT * 1e-3
    th = np.linspace(0, 180, 181)
    fig, ax = plt.subplots(figsize=(10, 4.0))
    fig.subplots_adjust(left=0.1, right=0.98, top=0.82, bottom=0.18)
    ax.plot(th, [dipole_energy(m, B, t) * 1e3 for t in th], color=VIOLET, label="U = −m B cos θ")
    ax.plot(th, [torque(m, B, t) * 1e3 for t in th], color=CORAL, label="torque = m B sin θ")
    U = energy_sum(theta, m, B)
    ax.plot([theta], [U * 1e3], "o", color=GOLD, ms=9)
    ax.set_xlabel("θ, the angle between m and B (degrees)"); ax.set_ylabel("mJ, or mN m"); ax.legend(frameon=False)
    fig.suptitle(f"at {theta:g}°: energy {U * 1e3:.3f} mJ (added up), torque {torque(m, B, theta) * 1e3:.3f} mN m",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig, "energy summed from 90° in 5,000 small turns"); plt.show()
    return U

assert close(magnet_picture(0.4, 20.0, 60.0), dipole_energy(0.4, 0.02, 60), 1e-6)
assert close(energy_sum(180, 0.02, 0.3) - energy_sum(0, 0.02, 0.3), 0.012, 1e-6) and agrees("P3.7", [0.012 / 0.6])

In [ ]:
interact(magnet_picture, m=FloatSlider(value=0.4, min=0.05, max=1.0, step=0.05, description="m (A m²)"),
         B_mT=FloatSlider(value=20.0, min=1.0, max=100.0, step=1.0, description="B (mT)"),
         theta=FloatSlider(value=60.0, min=0.0, max=180.0, step=5.0, description="θ (°)"));

## Sources
Every exercise and set-up here is our own. The constants are the course's rounded values (μ₀ = 4π × 10⁻⁷ T m/A, e and
the electron's and proton's masses from CODATA 2022, physics.nist.gov/cuu/Constants). The meter movement (50 µA full
scale, 2000 Ω) is the Simpson 260 volt-ohm-milliammeter's, from its operator's manual (1955); the two resistors are
computed here, and a real instrument's switched network differs in detail. The simulator is the course's: fourth-order
Runge–Kutta with a fixed step.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib, sys
_src = pathlib.Path("../../../episodes/ph16_magnetism/practice.py")
if _src.exists():
    sys.path.insert(0, str(_src.resolve().parents[2]))
    _spec = importlib.util.spec_from_file_location("ph16_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["parts"], _q["hint"]) == (
            _e["q"], _e["working"], _e["function"], _e["tier"], _e["parts"], _e["hint"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 * max(1, abs(b)) for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX)
    assert (MU0, KM, E, ME, MP, g) == (_P.MU0, _P.KM, _P.E, _P.ME, _P.MP, _P.g)
    assert close(_P.METER["S"], S_m, 1e-12) and close(_P.METER["R"], R_m, 1e-12) and close(_P.CIRC["r"], r_s, 1e-4)
    print("matches the course's practice set")